# Alpha Mask Benchmark — persistent model worker

结构：

```text
download_models.py   # 只负责下载，重复执行会复用本地文件
        │
        ▼
/kaggle/working/models/
        │
        ▼
model_runtime.py     # 加载 + warmup + inference，常驻 GPU
        │
        ▼
model_server.py      # localhost:7861
        │
        ▼
app.py               # 纯 Gradio/UI + HTTP 调用，不加载模型
```

**只重启 `app.py` 不会重新加载 GPU 模型。**
只有 `model_server.py` 被终止、Notebook Kernel 重启或 Kaggle Session 重启时才需要重新加载。


In [1]:
!python -m pip install -q -U uv

!uv pip install gradio-tunneling

!uv pip uninstall --system onnxruntime onnxruntime-gpu || true

!uv pip install --system \
    "pillow==11.3.0" \
    "onnxruntime-gpu==1.21.0" \
    "rembg[gpu]" \
    "gradio>=5" \
    "transformers>=4.48,<5" \
    "fastapi>=0.115" \
    "uvicorn>=0.30" \
    "python-multipart>=0.0.9" \
    "requests>=2.32" \
    timm einops kornia safetensors \
    "git+https://github.com/PramaLLC/BEN2.git"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 73.8 MB/s eta 0:00:00:00:0100:01
Using Python 3.12.13 environment at: /usr
Resolved 6 packages in 114ms                                         
Prepared 1 package in 329ms                                              
Installed 1 package in 3ms.9.0                              
 + gradio-tunneling==0.9.0
Using Python 3.12.13 environment at: /usr
Using Python 3.12.13 environment at: /usr
Resolved 111 packages in 2.73s                                       
Prepared 8 packages in 4.12s                                             
Uninstalled 2 packages in 510ms
Installed 8 packages in 56ms                                
 + ben2==0.0.1 (from git+https://github.com/PramaLLC/BEN2.git@2c99a5da477b5523585bfa5c893888a6e818a8f6)
 + coloredlogs==15.0.1
 - huggingface-hub==1.11.0
 + huggingface-hub==0.36.2
 + humanfriendly==10.0
 + onnxruntime-gpu==1.21.0
 + pymatting==1.1.15
 + rembg==2.0.69
 - transformers==5.0.0
 + transformers==4.57.6

In [2]:
import PIL
import torch
import onnxruntime as ort

try:
    ort.preload_dlls()
except Exception as e:
    print("ort.preload_dlls:", e)

print("Pillow:", PIL.__version__)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())
print("GPU 0:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
print("ONNX Runtime:", ort.__version__)
print("ORT providers:", ort.get_available_providers())

assert PIL.__version__ == "11.3.0"
assert torch.cuda.is_available(), "没有检测到 Kaggle GPU，请先打开 GPU。"
assert torch.cuda.device_count() >= 2, f"需要 Kaggle 2×T4，目前只有 {torch.cuda.device_count()} 张 GPU"
assert "CUDAExecutionProvider" in ort.get_available_providers(), "rembg 没有 CUDAExecutionProvider"


Pillow: 11.3.0
PyTorch: 2.10.0+cu128
CUDA: 12.8
GPU count: 2
GPU 0: Tesla T4
ONNX Runtime: 1.21.0
ORT providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [3]:
%%writefile download_models.py

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import snapshot_download, hf_hub_download
from huggingface_hub.utils import disable_progress_bars

disable_progress_bars()

ROOT = Path("/kaggle/working/models")
BEN2_DIR = ROOT / "ben2"
BIREF_DIR = ROOT / "birefnet"
U2NET_HOME = ROOT / "u2net"

for p in (BEN2_DIR, BIREF_DIR, U2NET_HOME):
    p.mkdir(parents=True, exist_ok=True)

def dl_ben2():
    required = BEN2_DIR / "model.safetensors"
    if not required.exists():
        snapshot_download(
            "PramaLLC/BEN2",
            local_dir=BEN2_DIR,
            allow_patterns=["config.json", "model.safetensors"],
        )
    return f"BEN2 OK -> {required}"

def dl_biref():
    required = BIREF_DIR / "model.safetensors"
    if not required.exists():
        snapshot_download(
            "ZhengPeng7/BiRefNet",
            local_dir=BIREF_DIR,
            allow_patterns=["config.json", "model.safetensors", "*.py"],
        )
    return f"BiRefNet OK -> {required}"

def dl_isnet():
    required = U2NET_HOME / "isnet-general-use.onnx"
    if not required.exists():
        hf_hub_download(
            "jellybox/isnet-general-use",
            "isnet-general-use.onnx",
            revision="407fc6fbe11da9209fa37b128c0fbbb03f29ad54",
            local_dir=U2NET_HOME,
        )
    return f"ISNet OK -> {required}"

with ThreadPoolExecutor(max_workers=3) as ex:
    for msg in ex.map(lambda fn: fn(), [dl_ben2, dl_biref, dl_isnet]):
        print(msg)


Writing download_models.py


In [4]:
!python download_models.py

BEN2 OK -> /kaggle/working/models/ben2/model.safetensors
BiRefNet OK -> /kaggle/working/models/birefnet/model.safetensors
ISNet OK -> /kaggle/working/models/u2net/isnet-general-use.onnx


In [5]:
%%writefile model_runtime.py

import os
import time
import threading
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

ROOT = Path("/kaggle/working/models")
HF_HOME = ROOT / "hf"
BEN2_DIR = ROOT / "ben2"
BIREF_DIR = ROOT / "birefnet"
U2NET_HOME = ROOT / "u2net"

HF_HOME.mkdir(parents=True, exist_ok=True)
U2NET_HOME.mkdir(parents=True, exist_ok=True)

# 必须在 transformers/rembg import 之前设置。
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["U2NET_HOME"] = str(U2NET_HOME)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import numpy as np
import torch
from PIL import Image
from torchvision import transforms
from transformers import AutoModelForImageSegmentation
from safetensors.torch import load_file
from ben2.modeling_ben2 import BEN_Base
from rembg import remove, new_session

if torch.cuda.device_count() < 2:
    raise RuntimeError(
        f"需要 Kaggle 2×T4，目前只检测到 {torch.cuda.device_count()} 张 GPU"
    )

BEN2_DEVICE = torch.device("cuda:0")
BIREF_DEVICE = torch.device("cuda:1")
REMBG_GPU = 0
REMBG_MODEL_NAME = "isnet-general-use"

GPU0_LOCK = threading.Lock()
GPU1_LOCK = threading.Lock()

BIREF_TRANSFORM = transforms.Compose([
    transforms.Resize((1024, 1024)),
    transforms.ToTensor(),
    transforms.Normalize([.485, .456, .406], [.229, .224, .225]),
])

def sync(device):
    torch.cuda.synchronize(device)

def prepare_image(image):
    if isinstance(image, np.ndarray):
        image = Image.fromarray(image)
    if not isinstance(image, Image.Image):
        raise TypeError(f"不支持的图片类型: {type(image)!r}")
    return image.convert("RGB")

def biref_tensor(image):
    return (
        BIREF_TRANSFORM(image.convert("RGB"))
        .unsqueeze(0)
        .to(BIREF_DEVICE, non_blocking=True)
    )

def gpu_status():
    lines = []
    for i in range(2):
        free, total = torch.cuda.mem_get_info(i)
        lines.append(
            f"cuda:{i} | {torch.cuda.get_device_name(i)} | "
            f"{(total-free)/2**30:.2f}/{total/2**30:.2f} GB"
        )
    return "\n".join(lines)

def _require(path):
    if not Path(path).exists():
        raise FileNotFoundError(
            f"缺少模型文件: {path}\n请先运行 download_models.py"
        )

_require(BEN2_DIR / "model.safetensors")
_require(BIREF_DIR / "model.safetensors")
_require(U2NET_HOME / "isnet-general-use.onnx")

# ============================================================
# Load once
# ============================================================

print("\n" + "=" * 70)
print("Loading BEN2 Base (LOCAL) -> cuda:0")
print("=" * 70)
t = time.perf_counter()

# 绕过 BEN2 AutoModel 的 model_info(repo_id) 网络查询，直接本地载入 safetensors。
BEN2_MODEL = BEN_Base()
BEN2_STATE = load_file(str(BEN2_DIR / "model.safetensors"), device="cpu")
BEN2_MODEL.load_state_dict(BEN2_STATE, strict=True)
del BEN2_STATE
BEN2_MODEL = BEN2_MODEL.to(BEN2_DEVICE).eval()
sync(BEN2_DEVICE)
print(f"BEN2 loaded: {time.perf_counter() - t:.2f}s")

print("\n" + "=" * 70)
print("Loading BiRefNet (LOCAL) -> cuda:1")
print("=" * 70)
t = time.perf_counter()
BIREF_MODEL = AutoModelForImageSegmentation.from_pretrained(
    str(BIREF_DIR),
    trust_remote_code=True,
    local_files_only=True,
).to(BIREF_DEVICE).eval()
sync(BIREF_DEVICE)
print(f"BiRefNet loaded: {time.perf_counter() - t:.2f}s")

print("\n" + "=" * 70)
print("Loading rembg/ISNet (LOCAL) -> cuda:0")
print("=" * 70)
t = time.perf_counter()
REMBG_SESSION = new_session(
    REMBG_MODEL_NAME,
    providers=[
        ("CUDAExecutionProvider", {"device_id": REMBG_GPU}),
        "CPUExecutionProvider",
    ],
)
print(f"rembg loaded: {time.perf_counter() - t:.2f}s")
print("providers:", REMBG_SESSION.inner_session.get_providers())

# BEN2 import 会主动改 cudnn 配置；模型构造完成后再恢复 benchmark。
torch.backends.cudnn.benchmark = True

# ============================================================
# Warmup once
# ============================================================

print("\n" + "=" * 70)
print("Warming up models...")
print("=" * 70)

warm = Image.new("RGB", (512, 512), (128, 128, 128))

print("Warmup BEN2...")
t = time.perf_counter()
with torch.inference_mode():
    _ = BEN2_MODEL.inference(warm)
sync(BEN2_DEVICE)
print(f"BEN2 warmup: {time.perf_counter() - t:.2f}s")

print("Warmup BiRefNet...")
x = biref_tensor(warm)
t = time.perf_counter()
with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
    _ = BIREF_MODEL(x)[-1]
sync(BIREF_DEVICE)
del x
print(f"BiRefNet warmup: {time.perf_counter() - t:.2f}s")

print("Warmup rembg...")
t = time.perf_counter()
_ = remove(warm, session=REMBG_SESSION)
print(f"rembg warmup: {time.perf_counter() - t:.2f}s")

del warm

print("\n" + "=" * 70)
print("ALL MODELS READY — model_runtime stays resident")
print("=" * 70)
print(gpu_status())
print("=" * 70)

# ============================================================
# Inference
# ============================================================

def _run_ben2(image):
    image = prepare_image(image)
    sync(BEN2_DEVICE)
    t = time.perf_counter()
    with torch.inference_mode():
        result = BEN2_MODEL.inference(image)
    sync(BEN2_DEVICE)
    dt = time.perf_counter() - t
    result = result.convert("RGBA")
    return (
        result,
        result.getchannel("A"),
        f"BEN2 Base\nGPU: cuda:0\n推理: {dt:.3f}s\n"
        f"输入: {image.width}×{image.height}",
    )

def run_ben2(image):
    with GPU0_LOCK:
        return _run_ben2(image)

def _run_rembg(image):
    image = prepare_image(image)
    t = time.perf_counter()
    result = remove(image, session=REMBG_SESSION).convert("RGBA")
    dt = time.perf_counter() - t
    return (
        result,
        result.getchannel("A"),
        f"ISNet / rembg\nGPU: cuda:0\n推理: {dt:.3f}s\n"
        f"输入: {image.width}×{image.height}",
    )

def run_rembg(image):
    with GPU0_LOCK:
        return _run_rembg(image)

def _run_birefnet(image):
    image = prepare_image(image)
    x = biref_tensor(image)

    sync(BIREF_DEVICE)
    t = time.perf_counter()

    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
        pred = BIREF_MODEL(x)[-1].sigmoid()

    sync(BIREF_DEVICE)
    dt = time.perf_counter() - t

    alpha = transforms.ToPILImage()(
        pred[0].squeeze().float().cpu()
    ).resize(image.size, Image.Resampling.LANCZOS)

    del x, pred

    result = image.convert("RGBA")
    result.putalpha(alpha)

    return (
        result,
        alpha,
        f"BiRefNet\nGPU: cuda:1\n推理: {dt:.3f}s\n"
        f"模型输入: 1024×1024\n原图: {image.width}×{image.height}",
    )

def run_birefnet(image):
    with GPU1_LOCK:
        return _run_birefnet(image)

def _gpu0(image):
    with GPU0_LOCK:
        return _run_ben2(image), _run_rembg(image)

def _gpu1(image):
    with GPU1_LOCK:
        return _run_birefnet(image)

def compare_all(image):
    image = prepare_image(image)
    t = time.perf_counter()

    with ThreadPoolExecutor(max_workers=2) as ex:
        f0 = ex.submit(_gpu0, image)
        f1 = ex.submit(_gpu1, image)
        ben2, rembg = f0.result()
        biref = f1.result()

    return (
        ben2,
        rembg,
        biref,
        f"总耗时: {time.perf_counter() - t:.3f}s\n\n{gpu_status()}",
    )


Writing model_runtime.py


In [6]:
%%writefile model_server.py

import base64
from io import BytesIO

from fastapi import FastAPI, File, HTTPException, UploadFile
from PIL import Image

# 这个 import 会加载并 warmup 一次；server 常驻期间不会再次执行。
from model_runtime import (
    compare_all,
    gpu_status,
    run_ben2,
    run_birefnet,
    run_rembg,
)

app = FastAPI(title="Persistent Alpha Model Worker")

def read_image(file: UploadFile):
    try:
        return Image.open(file.file).convert("RGB")
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"图片读取失败: {e}")

def encode_png(image):
    buf = BytesIO()
    image.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("ascii")

def pack(result):
    rgba, alpha, info = result
    return {
        "rgba": encode_png(rgba),
        "alpha": encode_png(alpha),
        "info": info,
    }

@app.get("/health")
def health():
    return {
        "ready": True,
        "gpu_status": gpu_status(),
    }

@app.post("/infer/ben2")
def infer_ben2(file: UploadFile = File(...)):
    return pack(run_ben2(read_image(file)))

@app.post("/infer/rembg")
def infer_rembg(file: UploadFile = File(...)):
    return pack(run_rembg(read_image(file)))

@app.post("/infer/birefnet")
def infer_birefnet(file: UploadFile = File(...)):
    return pack(run_birefnet(read_image(file)))

@app.post("/compare")
def compare(file: UploadFile = File(...)):
    ben2, rembg, biref, status = compare_all(read_image(file))
    return {
        "ben2": pack(ben2),
        "rembg": pack(rembg),
        "birefnet": pack(biref),
        "status": status,
    }

if __name__ == "__main__":
    import uvicorn

    # 必须保持 1 worker：多 worker 会复制一套 GPU 模型。
    uvicorn.run(
        app,
        host="127.0.0.1",
        port=7861,
        workers=1,
        log_level="info",
    )


Writing model_server.py


In [11]:
%%writefile app.py
import base64, html, json, math, time, uuid
from io import BytesIO
from pathlib import Path
import gradio as gr
import requests
from PIL import Image

WORKER = "http://127.0.0.1:7861"
TIMEOUT = 300
OUTPUT_ROOT = Path("/kaggle/working/alpha_outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DEFAULT_PAGE_SIZE = 4

def worker_status():
    try:
        r = requests.get(f"{WORKER}/health", timeout=3); r.raise_for_status(); data = r.json()
        return "✅ Model Worker READY\n" + data.get("gpu_status", "")
    except Exception as e:
        return f"❌ Model Worker NOT READY\n{e}"

def image_to_bytes(image):
    buf = BytesIO(); image.convert("RGB").save(buf, format="PNG"); return buf.getvalue()

def call_compare(image):
    r = requests.post(f"{WORKER}/compare", files={"file": ("input.png", image_to_bytes(image), "image/png")}, timeout=TIMEOUT)
    r.raise_for_status(); return r.json()

def decode_png(data):
    return Image.open(BytesIO(base64.b64decode(data))).copy()

def save_payload(payload, output_dir, name):
    rgba_path, alpha_path = output_dir / f"{name}.png", output_dir / f"{name}_alpha.png"
    decode_png(payload["rgba"]).save(rgba_path); decode_png(payload["alpha"]).save(alpha_path)
    return {"image": str(rgba_path), "alpha": str(alpha_path), "info": payload.get("info", "")}

def save_manifest(run_dir, records):
    path, tmp = run_dir / "manifest.json", run_dir / "manifest.tmp"
    tmp.write_text(json.dumps({"run_dir": str(run_dir), "updated_at": time.time(), "records": records}, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp.replace(path)

def latest_manifest():
    manifests = list(OUTPUT_ROOT.glob("run_*/manifest.json"))
    return max(manifests, key=lambda p: p.stat().st_mtime) if manifests else None

def load_latest():
    path = latest_manifest()
    if path is None: return [], ""
    try:
        data = json.loads(path.read_text(encoding="utf-8")); return data.get("records", []), data.get("run_dir", str(path.parent))
    except Exception:
        return [], ""

def image_data_uri(path, max_side=640):
    path = Path(path)
    if not path.exists(): return ""
    with Image.open(path) as img:
        img = img.convert("RGBA"); img.thumbnail((max_side, max_side), Image.Resampling.LANCZOS); buf = BytesIO(); img.save(buf, format="PNG", optimize=True)
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode("ascii")

def empty_html():
    return """<div class="empty-state"><div class="empty-icon">▧</div><div class="empty-title">暂无处理结果</div><div class="empty-text">上传图片后，BEN2 / ISNet / BiRefNet 会自动同时运行。</div></div>"""

def model_figure(title, path, info=""):
    if not path or not Path(path).exists(): return f'<div class="model-result"><div class="model-title">{html.escape(title)}</div><div class="missing-image">无输出</div></div>'
    uri = image_data_uri(path); info_html = html.escape(info).replace("\n", "<br>")
    return f'<div class="model-result"><div class="model-title">{html.escape(title)}</div><div class="image-stage"><img src="{uri}" loading="lazy"></div>{f"""<div class="model-info">{info_html}</div>""" if info else ""}</div>'

def render_page(records, page, page_size):
    if not records: return empty_html()
    page_size = int(page_size); total_pages = max(1, math.ceil(len(records) / page_size)); page = max(1, min(int(page), total_pages)); start = (page - 1) * page_size; end = min(start + page_size, len(records)); cards = []
    for record in records[start:end]:
        index, name, error = record.get("index", 0), html.escape(record.get("name", "image")), record.get("error")
        if error:
            cards.append(f'<article class="sample-card error-card"><div class="sample-header"><div><span class="sample-index">#{index:03d}</span><span class="sample-name">{name}</span></div><span class="error-badge">ERROR</span></div><div class="error-message">{html.escape(str(error))}</div></article>'); continue
        original = model_figure("Original", record["input"]); ben2 = model_figure("BEN2", record["ben2"]["image"], record["ben2"].get("info", "")); isnet = model_figure("ISNet", record["isnet"]["image"], record["isnet"].get("info", "")); birefnet = model_figure("BiRefNet", record["birefnet"]["image"], record["birefnet"].get("info", ""))
        cards.append(f'<article class="sample-card"><div class="sample-header"><div><span class="sample-index">#{index:03d}</span><span class="sample-name">{name}</span></div></div><div class="compare-grid">{original}{ben2}{isnet}{birefnet}</div></article>')
    return '<div class="results-container">' + "".join(cards) + "</div>"

def page_label(records, page, page_size):
    if not records: return "0 / 0"
    page_size = int(page_size); total_pages = max(1, math.ceil(len(records) / page_size)); page = max(1, min(int(page), total_pages)); start = (page - 1) * page_size + 1; end = min(page * page_size, len(records))
    return f"第 {page} / {total_pages} 页　·　{start}–{end} / {len(records)}"

def normalize_file_path(file):
    if isinstance(file, (str, Path)): return Path(file)
    if hasattr(file, "name"): return Path(file.name)
    if hasattr(file, "path"): return Path(file.path)
    raise TypeError(f"未知文件类型: {type(file)}")

def process_batch(files, page_size, progress=gr.Progress()):
    if not files: raise gr.Error("请先上传至少一张图片")
    run_id = time.strftime("run_%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6]; run_dir = OUTPUT_ROOT / run_id; run_dir.mkdir(parents=True, exist_ok=True); records = []; total = len(files)
    for i, file in enumerate(files, start=1):
        progress((i - 1, total), desc=f"处理中 {i}/{total}"); source_path = normalize_file_path(file); record = {"index": i, "name": source_path.name}; item_dir = run_dir / f"{i:04d}_{source_path.stem[:60]}"; item_dir.mkdir(parents=True, exist_ok=True)
        try:
            with Image.open(source_path) as img: image = img.convert("RGB")
            input_path = item_dir / "input.png"; image.save(input_path); result = call_compare(image); record["input"] = str(input_path)
            record["ben2"] = save_payload(result["ben2"], item_dir, "ben2"); record["isnet"] = save_payload(result["rembg"], item_dir, "isnet"); record["birefnet"] = save_payload(result["birefnet"], item_dir, "birefnet"); record["benchmark"] = result.get("status", "")
        except Exception as e:
            record["error"] = f"{type(e).__name__}: {e}"
        records.append(record); save_manifest(run_dir, records)
    progress((total, total), desc="完成"); success = sum(1 for r in records if not r.get("error")); failed = len(records) - success; page = 1
    status = f"{worker_status()}\n\n本次任务：{len(records)} 张\n成功：{success}\n失败：{failed}\n\n保存目录：\n{run_dir}"
    return records, page, render_page(records, page, page_size), page_label(records, page, page_size), status, str(run_dir)

def previous_page(records, page, page_size):
    page = max(1, int(page) - 1); return page, render_page(records, page, page_size), page_label(records, page, page_size)

def next_page(records, page, page_size):
    if not records: return 1, empty_html(), "0 / 0"
    page_size = int(page_size); total_pages = max(1, math.ceil(len(records) / page_size)); page = min(total_pages, int(page) + 1)
    return page, render_page(records, page, page_size), page_label(records, page, page_size)

def change_page_size(records, page_size):
    return 1, render_page(records, 1, page_size), page_label(records, 1, page_size)

def reload_latest(page_size):
    records, run_dir = load_latest(); page = 1; status = f"{worker_status()}\n\n" + (f"已载入最近任务：\n{run_dir}" if records else "Kaggle 本地暂无历史任务。")
    return records, page, render_page(records, page, page_size), page_label(records, page, page_size), status, run_dir

INITIAL_RECORDS, INITIAL_RUN = load_latest()

CSS = """
.gradio-container{max-width:1600px!important;margin:0 auto!important}
.results-container{display:flex;flex-direction:column;gap:20px}
.sample-card{border:1px solid rgba(128,128,128,.22);border-radius:16px;overflow:hidden;background:var(--background-fill-primary)}
.sample-header{display:flex;justify-content:space-between;align-items:center;padding:13px 16px;border-bottom:1px solid rgba(128,128,128,.18)}
.sample-index{font-family:ui-monospace,monospace;font-size:12px;opacity:.58;margin-right:8px}
.sample-name{font-weight:600;font-size:14px}
.compare-grid{display:grid;grid-template-columns:repeat(4,minmax(0,1fr))}
.model-result{min-width:0;border-right:1px solid rgba(128,128,128,.14)}
.model-result:last-child{border-right:none}
.model-title{height:40px;display:flex;align-items:center;padding:0 12px;font-weight:650;font-size:13px;border-bottom:1px solid rgba(128,128,128,.14)}
.image-stage{height:330px;display:flex;align-items:center;justify-content:center;padding:10px;background-color:#f8f8f8;background-image:linear-gradient(45deg,#e8e8e8 25%,transparent 25%),linear-gradient(-45deg,#e8e8e8 25%,transparent 25%),linear-gradient(45deg,transparent 75%,#e8e8e8 75%),linear-gradient(-45deg,transparent 75%,#e8e8e8 75%);background-size:20px 20px;background-position:0 0,0 10px,10px -10px,-10px 0}
.image-stage img{width:100%;height:100%;object-fit:contain}
.model-info{padding:9px 12px 11px;min-height:48px;font-size:11px;line-height:1.45;opacity:.65;font-family:ui-monospace,monospace}
.empty-state{min-height:360px;display:flex;flex-direction:column;align-items:center;justify-content:center;border:1px dashed rgba(128,128,128,.35);border-radius:16px}
.empty-icon{font-size:38px;opacity:.35}.empty-title{font-weight:650;margin-top:12px}.empty-text{margin-top:5px;font-size:13px;opacity:.58}
.error-message{margin:14px;font-family:ui-monospace,monospace;font-size:12px}
@media(max-width:1150px){.compare-grid{grid-template-columns:repeat(2,minmax(0,1fr))}}
@media(max-width:720px){.compare-grid{grid-template-columns:1fr}.image-stage{height:280px}}
"""

with gr.Blocks(title="Alpha Batch Benchmark", css=CSS) as demo:
    records_state = gr.State(INITIAL_RECORDS); page_state = gr.State(1)
    gr.Markdown("# Alpha Batch Benchmark\n批量上传图片，自动同时运行 **BEN2 / ISNet / BiRefNet**。完整结果自动保存到 Kaggle 本地。")
    with gr.Row():
        with gr.Column(scale=3):
            files = gr.File(label="批量上传图片", file_count="multiple", file_types=["image"], type="filepath")
            with gr.Row():
                process_btn = gr.Button("开始批量处理", variant="primary", scale=3); reload_btn = gr.Button("载入最近任务", scale=1)
        with gr.Column(scale=2):
            status = gr.Textbox(label="运行状态", value=worker_status() + (f"\n\n最近任务：\n{INITIAL_RUN}" if INITIAL_RUN else ""), lines=6, interactive=False)
            run_path = gr.Textbox(label="Kaggle 保存目录", value=INITIAL_RUN, interactive=False)

    gr.Markdown("## Results")
    with gr.Row():
        prev_btn = gr.Button("← 上一页"); page_text = gr.Textbox(value=page_label(INITIAL_RECORDS, 1, DEFAULT_PAGE_SIZE), show_label=False, interactive=False, scale=2); page_size = gr.Dropdown([4, 8, 12], value=DEFAULT_PAGE_SIZE, label="每页"); next_btn = gr.Button("下一页 →")
    results = gr.HTML(value=render_page(INITIAL_RECORDS, 1, DEFAULT_PAGE_SIZE))

    process_btn.click(process_batch, [files, page_size], [records_state, page_state, results, page_text, status, run_path])
    reload_btn.click(reload_latest, page_size, [records_state, page_state, results, page_text, status, run_path])
    prev_btn.click(previous_page, [records_state, page_state, page_size], [page_state, results, page_text])
    next_btn.click(next_page, [records_state, page_state, page_size], [page_state, results, page_text])
    page_size.change(change_page_size, [records_state, page_size], [page_state, results, page_text])

if __name__ == "__main__":
    demo.queue(default_concurrency_limit=1).launch(server_name="127.0.0.1", server_port=7860, share=False, show_error=True)

Overwriting app.py


## 启动常驻模型 Worker

这个 Cell 可以重复运行：

- `7861` 已经有健康 Worker：**直接复用，不重新加载模型**
- Worker 不存在：才启动 `model_server.py` 并加载 / warmup


In [12]:
import subprocess
import sys
import time
from pathlib import Path

import requests

WORKER_URL = "http://127.0.0.1:7861/health"
WORKER_LOG_PATH = Path("/kaggle/working/model_server.log")

def worker_ready():
    try:
        r = requests.get(WORKER_URL, timeout=1)
        return r.ok and r.json().get("ready") is True
    except Exception:
        return False

if worker_ready():
    print("✅ 复用现有 model_server.py；不会重新加载模型")
else:
    WORKER_LOG = open(WORKER_LOG_PATH, "a", buffering=1)
    MODEL_SERVER_PROCESS = subprocess.Popen(
        [sys.executable, "model_server.py"],
        stdout=WORKER_LOG,
        stderr=subprocess.STDOUT,
        cwd="/kaggle/working",
    )

    while not worker_ready():
        if MODEL_SERVER_PROCESS.poll() is not None:
            WORKER_LOG.flush()
            print(WORKER_LOG_PATH.read_text(errors="replace")[-8000:])
            raise RuntimeError(
                f"model_server.py 已退出，code={MODEL_SERVER_PROCESS.returncode}"
            )
        time.sleep(1)

    print("✅ model_server.py READY")

print(requests.get(WORKER_URL, timeout=3).json()["gpu_status"])


✅ model_server.py READY
cuda:0 | Tesla T4 | 3.94/14.56 GB
cuda:1 | Tesla T4 | 3.67/14.56 GB


## 启动 / 重启 Gradio UI

这个 Cell 只重启 `app.py`。  
**不会动 7861 的模型 Worker，因此不会重新加载 BEN2 / BiRefNet / ISNet。**


In [13]:
import subprocess, sys, time
from pathlib import Path
import requests

APP_LOG_PATH = Path("/kaggle/working/app.log")

if "APP_PROCESS" in globals():
    try:
        if APP_PROCESS.poll() is None: APP_PROCESS.terminate(); APP_PROCESS.wait(timeout=5)
    except Exception:
        try: APP_PROCESS.kill()
        except Exception: pass

APP_LOG = open(APP_LOG_PATH, "a", buffering=1)
APP_PROCESS = subprocess.Popen([sys.executable, "app.py"], stdout=APP_LOG, stderr=subprocess.STDOUT, cwd="/kaggle/working")

while True:
    if APP_PROCESS.poll() is not None:
        APP_LOG.flush(); print(APP_LOG_PATH.read_text(errors="replace")[-8000:]); raise RuntimeError(f"app.py 已退出，code={APP_PROCESS.returncode}")
    try:
        if requests.get("http://127.0.0.1:7860", timeout=1).ok: break
    except Exception: pass
    time.sleep(0.5)

print("✅ Gradio ready: http://127.0.0.1:7860")
print("✅ model_server.py 未重启，BEN2 / ISNet / BiRefNet 继续常驻 GPU")

✅ Gradio ready: http://127.0.0.1:7860
✅ model_server.py 未重启，BEN2 / ISNet / BiRefNet 继续常驻 GPU


In [ ]:
!gradio-tun 7860

公网访问地址：https://1be38c59b5843de4c9.gradio.live
这个共享链接将在 72 小时后过期，此程序将在 72 小时后关闭。
